## Imports and configuration

Import required libraries (numpy, pandas, urllib.parse, Counter, matplotlib), set the path to the URL dataset and column names, and create the outputs directory for saving train/test CSVs.

In [ ]:
import os
import numpy as np
import pandas as pd
from urllib.parse import urlparse
from collections import Counter
import matplotlib.pyplot as plt


DATA_PATH = "../data/urldata.csv"  
URL_COL   = "url"
LABEL_COL = "label"                      

os.makedirs("outputs", exist_ok=True)


## Load dataset

Check that the CSV exists at DATA_PATH, load it into a DataFrame, and display the column names, shape, and first few rows.

In [3]:
assert os.path.exists(DATA_PATH), f"CSV not found at {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print("Columns:", df.columns.tolist(), "Shape:", df.shape)
df.head()


Columns: ['number', 'url', 'label', 'result'] Shape: (450176, 4)


,number,url,label,result
0,0,https://www.google.com,benign,0
1,1,https://www.youtube.com,benign,0
2,2,https://www.facebook.com,benign,0
3,3,https://www.baidu.com,benign,0
4,4,https://www.wikipedia.org,benign,0


## Cleaning and binary labels

Convert text labels to binary (0 = benign, 1 = malicious), drop rows with missing URL or label, remove empty URLs, and remove duplicate URLs. Print the shape after cleaning and the label distribution.

In [4]:
# Convert labels to binary: 1 = malicious, 0 = benign

if LABEL_COL in df.columns:
    df["label"] = df["label"].map({"benign": 0, "malicious": 1}).fillna(df["label"]).astype(int)
else:
    raise ValueError("No label column found. Set LABEL_COL correctly.")

# Remove empty rows
df = df.dropna(subset=[URL_COL, "label"])
# Remove empty URLs
df = df[df[URL_COL].astype(str).str.strip() != ""]
# Remove duplicate URLs
df = df.drop_duplicates(subset=[URL_COL]).reset_index(drop=True)

print("After cleaning:", df.shape)
df["label"].value_counts()


After cleaning: (450176, 4)


label
0    345738
1    104438
Name: count, dtype: int64

## URL standardization

Define standard_url() to lowercase URLs, add http:// when no scheme is present, parse with urlparse, and normalize netloc (strip port) and path/query. Apply it to create a url_clean column and show a sample.

In [ ]:
# Standardize URLs for better feature extraction
def standard_url(u: str) -> str:
    try:
        # Convert to string and normalize case
        u = str(u).strip().lower()
        # Check for empty URLs
        if not u:
            return ""
        if not u.startswith(("http://", "https://", "ftp://")):
            u = "http://" + u
        # Parse the URL
        p = urlparse(u)
        # Extract the netloc and strip any ports
        netloc = p.netloc.split(":")[0] if p.netloc else ""  
        # If parsing failed completely, return original URL
        if not netloc and not p.path:
            return u  
        return f"{p.scheme}://{netloc}{p.path}" + (f"?{p.query}" if p.query else "")
    except (ValueError, Exception):
        return str(u).strip().lower()

df["url_clean"] = df[URL_COL].apply(standard_url)
df[[URL_COL, "url_clean"]].head()


,url,url_clean
0,https://www.google.com,https://www.google.com
1,https://www.youtube.com,https://www.youtube.com
2,https://www.facebook.com,https://www.facebook.com
3,https://www.baidu.com,https://www.baidu.com
4,https://www.wikipedia.org,https://www.wikipedia.org


## Feature extraction

Define Shannon entropy and extract_features(): for each URL compute length (url, host, path, query), digit/special/uppercase counts and ratios, subdomain count, entropy, and is_https. Apply to all URLs and show the resulting 13-column feature matrix.

In [ ]:
SPECIALS = "-_@?%="

# Define the shannon entropy function
def shannon_entropy(s: str) -> float:

    if not s: return 0.0
    counts = Counter(s)
    probs = np.fromiter((c/len(s) for c in counts.values()), dtype=float)
    return float(-(probs * np.log2(probs)).sum())

# Define the extract features function
def extract_features(original_url: str) -> pd.Series:
    try:
        
        url = standard_url(original_url)
        try:
            p = urlparse(url)
            host, path, query = p.netloc or "", p.path or "", p.query or ""
            is_https = 1 if p.scheme == "https" else 0
        except (ValueError, Exception):
            host, path, query = "", "", ""
            is_https = 1 if url.startswith("https://") else 0

        # Extract features
        total_len = len(url)
        count_digits   = sum(ch.isdigit() for ch in url)
        count_specials = sum(ch in SPECIALS for ch in url)
        count_upper    = sum(ch.isupper() for ch in str(original_url))
        num_subdomains = max(0, host.count(".") - 1) if host else 0

        return pd.Series({
            "url_length": total_len,
            "host_length": len(host),
            "path_length": len(path),
            "query_length": len(query),

            "count_digits": count_digits,
            "count_specials": count_specials,
            "count_upper": count_upper,
            "count_subdomains": num_subdomains,

            "digit_ratio": (count_digits/total_len) if total_len else 0.0,
            "symbol_ratio": (count_specials/total_len) if total_len else 0.0,
            "uppercase_ratio": (count_upper/len(str(original_url))) if len(str(original_url)) else 0.0,

            "entropy": shannon_entropy(url),
            "is_https": is_https
        })
    except Exception as e:
        print(f"Error extracting features from {original_url}: {e}")

features = df[URL_COL].apply(extract_features)
print("Feature shape:", features.shape)
features.head()


Feature shape: (450176, 13)


,url_length,host_length,path_length,query_length,count_digits,count_specials,count_upper,count_subdomains,digit_ratio,symbol_ratio,uppercase_ratio,entropy,is_https
0,22.0,14.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.663533,1.0
1,23.0,15.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.762267,1.0
2,24.0,16.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.855389,1.0
3,21.0,13.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.880180,1.0
4,25.0,17.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.813661,1.0


## Train/test split and scaling

Build the feature matrix X_raw and labels y, then split with train_test_split (80/20, stratified, random_state=42). Fit StandardScaler on the training numeric columns only and transform both train and test; concatenate scaled numerics with the categorical is_https to form X_train and X_test.

In [ ]:
# Numeric features
num_cols = [
    "url_length","host_length","path_length","query_length",
    "count_digits","count_specials","count_upper","count_subdomains",
    "digit_ratio","symbol_ratio","uppercase_ratio","entropy"
]
cat_cols = ["is_https"]

# Combine numeric & categorical features
X_raw = pd.concat([features[num_cols], features[cat_cols]], axis=1)
y = df["label"].astype(int)


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)
# Scale numerics using only training stats to prevent feature dominance 
scaler = StandardScaler().fit(X_train_raw[num_cols])
# Scale the numeric features of the training set
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train_raw[num_cols]),
    columns=num_cols,
    index=X_train_raw.index
)
# Scale the numeric features of the test set    
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw[num_cols]),
    columns=num_cols,
    index=X_test_raw.index
)

X_train = pd.concat([X_train_scaled, X_train_raw[cat_cols]], axis=1)
X_test  = pd.concat([X_test_scaled,  X_test_raw[cat_cols]], axis=1)

print("Train:", X_train.shape, "Test:", X_test.shape)



Train: (360140, 13) Test: (90036, 13)


## Save train/test CSVs

Ensure outputs exists and write X_train.csv, X_test.csv, y_train.csv, and y_test.csv so the model notebooks can load them.

In [ ]:
os.makedirs("outputs", exist_ok=True)
X_train.to_csv("outputs/X_train.csv", index=False)
X_test.to_csv("outputs/X_test.csv", index=False)
y_train.to_csv("outputs/y_train.csv", index=False)
y_test.to_csv("outputs/y_test.csv", index=False)

Saved scaled data & scaler stats in outputs/
